# NB15 — Descriptor dump (extract once on GPU, analyse locally)

Extracts L2-normalised global descriptors (fp16) + pose metadata for the reliability study, so that every
downstream experiment runs on CPU from the dumped files:

- **RobotCar-Seasons v2** — map = `overcast-reference` **rear** camera with poses from `3D-models/all-merged/all.nvm`
  (NVM: world→camera quaternion + camera centre); queries = the 1906 rear images of `robotcar_v2_train.txt`
  (4×4 **camera→world**, verified: motion ∥ optical axis, |cos|=0.98). Official thresholds (0.25 m, 2°), (0.5 m, 5°),
  (5 m, 10°) can then be applied to top-1 reference poses.
- **Extended-CMU-Seasons** — per slice: official database (`ground-truth-database-images-sliceN.txt`) as map, every
  dated query file in `camera-poses/` as queries (camera c0; xyz = camera centre, quaternion world→camera).
- **Nordland** — 4 seasons, frame-aligned, uniform subsample.

Outputs go to `/kaggle/working/dump/<descriptor>/`.

In [ ]:
# === P0.0 — imports / device ==========================================================
import os, sys, json, time, glob, math, warnings
from dataclasses import dataclass, asdict
from collections import OrderedDict
import numpy as np
import torch
REPORT = OrderedDict()
def log(step, status, msg=""):
    REPORT[step] = {"status": status, "msg": msg}
    print({"ok": "[ OK ]", "warn": "[WARN]", "fail": "[FAIL]", "info": "[info]"}.get(status, "[??]"), step + ":", msg, flush=True)
torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BF16_OK = (DEVICE == "cuda" and torch.cuda.is_bf16_supported())
print("torch", torch.__version__, "| device", DEVICE, "| bf16", BF16_OK, flush=True)
if DEVICE == "cuda": print("GPU:", torch.cuda.get_device_properties(0).name, flush=True)

In [ ]:
# === P0.1 — CONFIG ======================================================================
@dataclass
class Config:
    INPUT_ROOTS: tuple = ("/kaggle/input",)
    OUT_DIR: str = "/kaggle/working/dump"
    IMG_SIZE: int = 384
    BATCH_SIZE: int = 64
    NUM_WORKERS: int = 4
    USE_BF16: bool = True
    NORD_PER_SEASON: int = 4500
    CMU_MAX_IMAGES: int = 24000
    CMU_SLICE_ORDER: tuple = (7, 10, 19, 8, 11, 20, 6, 12, 21, 5, 13, 22, 4, 14, 23, 3, 15, 24, 2, 16, 17, 18, 9, 25)
CONFIG = Config()
CONFIG.USE_BF16 = CONFIG.USE_BF16 and BF16_OK
if DEVICE == "cuda":
    g = torch.cuda.get_device_properties(0).total_memory / 1e9
    CONFIG.BATCH_SIZE = 256 if g >= 70 else 64
os.makedirs(CONFIG.OUT_DIR, exist_ok=True)
print(asdict(CONFIG), flush=True)

In [ ]:
# === P0.2 — offline env + path discovery ===============================================
for k, v in {"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "PIP_NO_INDEX": "1", "WANDB_MODE": "offline"}.items():
    os.environ.setdefault(k, v)

def _find_dir(markers_any, markers_all=(), roots=None, max_depth=4):
    roots = roots or CONFIG.INPUT_ROOTS; seen = set()
    for root in roots:
        if not os.path.isdir(root): continue
        frontier = [(root, 0)]
        while frontier:
            d, dep = frontier.pop(0)
            if d in seen or dep > max_depth: continue
            seen.add(d)
            try: entries = set(os.listdir(d))
            except Exception: continue
            ok_all = all(any(m.lower() in e.lower() for e in entries) for m in markers_all) if markers_all else True
            ok_any = any(any(m.lower() in e.lower() for e in entries) for m in markers_any) if markers_any else True
            if ok_all and ok_any: return d
            for e in sorted(entries):
                sub = os.path.join(d, e)
                if os.path.isdir(sub): frontier.append((sub, dep + 1))
    return None

PATHS = {
    "bundle":   _find_dir(["checkpoints"], ["checkpoints", "sources"]),
    "robotcar": _find_dir(["images"], ["images", "3D-models"]),
    "nordland": _find_dir(["data"], ["data", "annotations"]),
}
for n, p in PATHS.items(): log(f"P0.2.path.{n}", "ok" if p else "warn", p or "not found")
if PATHS["bundle"]:
    for sub in ["salad", "dinov2", ""]:
        p = os.path.join(PATHS["bundle"], "sources", sub) if sub else os.path.join(PATHS["bundle"], "sources")
        if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p)
print("inputs:", sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else None, flush=True)

In [ ]:
DUMP = ["rc", "nord", "cmu"]
DESCRIPTORS = ["jist"]


## Extractor registry + fp16 batch extraction

In [ ]:
# === E.0 — descriptor registry ===========================================================
EXTRACTOR_INFO = {"name": None, "dim": None}

def _try_salad():
    """serizba/salad via helper.get_* with the released dino_salad.ckpt config (dim 8448)."""
    ck = os.path.join(PATHS["bundle"], "checkpoints", "dino_salad.ckpt")
    salad_dir = os.path.join(PATHS["bundle"], "sources", "salad")
    if not (os.path.isfile(ck) and os.path.isdir(salad_dir)): return None
    if salad_dir not in sys.path: sys.path.insert(0, salad_dir)
    from models import helper
    backbone = helper.get_backbone("dinov2_vitb14", {"num_trainable_blocks": 4, "return_token": True, "norm_layer": True})
    aggregator = helper.get_aggregator("SALAD", {"num_channels": 768, "num_clusters": 64, "cluster_dim": 128, "token_dim": 256})
    sd = torch.load(ck, map_location="cpu"); sd = sd.get("state_dict", sd)
    backbone.load_state_dict({k[9:]: v for k, v in sd.items() if k.startswith("backbone.")}, strict=False)
    aggregator.load_state_dict({k[11:]: v for k, v in sd.items() if k.startswith("aggregator.")}, strict=False)
    m = torch.nn.Module(); m.backbone = backbone; m.aggregator = aggregator; m.eval().to(DEVICE)
    @torch.no_grad()
    def fwd(x): return m.aggregator(m.backbone(x))
    return fwd

def _try_dinov2():
    pth = os.path.join(PATHS["bundle"], "checkpoints", "dinov2_vitb14_pretrain.pth")
    vit_base = __import__("dinov2.models.vision_transformer", fromlist=["vit_base"]).vit_base
    vit = vit_base(patch_size=14, img_size=518, init_values=1.0, block_chunks=0)
    sd = torch.load(pth, map_location="cpu"); vit.load_state_dict(sd.get("state_dict", sd), strict=False)
    vit.eval().to(DEVICE)
    @torch.no_grad()
    def fwd(x):
        f = vit.forward_features(x)
        return torch.cat([f["x_norm_clstoken"], f["x_norm_patchtokens"].mean(1)], 1)
    return fwd

REGISTRY = {"salad": _try_salad, "dinov2": _try_dinov2}

def build_extractor(name):
    fwd = REGISTRY[name]()
    assert fwd is not None, f"extractor {name} unavailable"
    with torch.no_grad():
        d = fwd(torch.zeros(2, 3, CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, device=DEVICE))
    assert torch.isfinite(d).all() and d.ndim == 2
    EXTRACTOR_INFO.update(name=name, dim=int(d.shape[1]))
    log("E.0.extractor", "ok", f"{name} dim={d.shape[1]}")
    return fwd

In [ ]:
# === E.1 — image dataset + fp16 extraction ================================================
from PIL import Image
import torchvision.transforms as T

class ImageList(torch.utils.data.Dataset):
    def __init__(self, paths, sz):
        self.paths = paths; self.sz = sz
        self.tx = T.Compose([T.Resize((480, 640), antialias=True), T.ToTensor(),
                             T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try: im = Image.open(self.paths[i]).convert("RGB"); ok = 1
        except Exception: im = Image.new("RGB", (self.sz, self.sz)); ok = 0
        return self.tx(im), i, ok

THROUGHPUT = []
@torch.no_grad()
def extract(paths, fwd, tag):
    dl = torch.utils.data.DataLoader(ImageList(paths, CONFIG.IMG_SIZE), batch_size=CONFIG.BATCH_SIZE, shuffle=False,
                                     num_workers=CONFIG.NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    out = np.empty((len(paths), EXTRACTOR_INFO["dim"]), np.float16); bad = 0; t0 = time.time()
    amp = torch.bfloat16 if CONFIG.USE_BF16 else torch.float16
    for b, (x, idx, ok) in enumerate(dl):
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=amp, enabled=(DEVICE == "cuda")):
            d = fwd(x)
        out[idx.numpy()] = torch.nn.functional.normalize(d.float(), dim=1).cpu().numpy().astype(np.float16)
        bad += int((ok == 0).sum())
        if b % 20 == 0: print(f"    {tag}: {min((b+1)*CONFIG.BATCH_SIZE, len(paths))}/{len(paths)} "
                              f"({(b+1)*CONFIG.BATCH_SIZE/max(time.time()-t0,1e-3):.1f} img/s)", flush=True)
    dt = time.time() - t0; THROUGHPUT.append((tag, len(paths), dt))
    assert np.isfinite(out.astype(np.float32)).all(), f"non-finite descriptors in {tag}"
    log(f"E.1.{tag}", "ok" if bad == 0 else "warn", f"{out.shape} in {dt:.0f}s ({len(paths)/max(dt,1e-3):.1f} img/s), unreadable={bad}")
    return out

def quat_to_R(q):
    w, x, y, z = q
    return np.array([[1-2*(y*y+z*z), 2*(x*y-z*w), 2*(x*z+y*w)],
                     [2*(x*y+z*w), 1-2*(x*x+z*z), 2*(y*z-x*w)],
                     [2*(x*z-y*w), 2*(y*z+x*w), 1-2*(x*x+y*y)]])

def rot_angle_deg(Ra, Rb):
    c = (np.trace(Ra.T @ Rb) - 1) / 2
    return float(np.degrees(np.arccos(np.clip(c, -1, 1))))

In [ ]:
# === E.2 — extra descriptors from the `vpr-descriptor-weights` dataset (MixVPR / CosPlace / NetVLAD / MegaLoc) ===
def _find_file_dir(fname, roots=("/kaggle/input",), max_depth=4):
    for root in roots:
        for d, subs, files in os.walk(root):
            if d.count(os.sep) - root.count(os.sep) >= max_depth: subs[:] = []
            if fname in files: return d
    return None

WROOT = _find_file_dir("vpr_extra.py")
if WROOT:
    if WROOT not in sys.path: sys.path.insert(0, WROOT)
    import vpr_extra
    for _n, _b in vpr_extra.BUILDERS.items():
        REGISTRY[_n] = (lambda b=_b: b(WROOT, DEVICE))
    print("extra descriptors available:", list(vpr_extra.BUILDERS), "| weights:",
          {d: sorted(os.listdir(os.path.join(WROOT, d))) for d in vpr_extra.BUILDERS if os.path.isdir(os.path.join(WROOT, d))}, flush=True)
else:
    print("vpr-descriptor-weights not attached -> only bundle descriptors", flush=True)

In [ ]:

# === J.0 — JIST (Berton et al., RA-L 2024; MIT code) — downloaded with internet ON (user-approved 28/11/2026) ===
import subprocess, re as _re
os.environ.pop("PIP_NO_INDEX", None)          # must be removed BEFORE pip (set earlier for offline runs)
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "einops"], capture_output=True, text=True)
print("pip:", _r.returncode, _r.stderr[-300:], flush=True)
JD = "/kaggle/working/JIST"
if not os.path.isdir(JD): subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ga1i13o/JIST", JD], check=True)
print("JIST commit:", subprocess.run(["git", "-C", JD, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(), flush=True)
DEF = {}
for f in subprocess.run(["grep", "-rn", "-E", "img_shape|seq_length|fc_output_dim|backbone", JD + "/jist/utils"], capture_output=True, text=True).stdout.splitlines()[:40]:
    print("  ", f[:200], flush=True)
CK = "/kaggle/working/jist_ckpt.pth"
if not os.path.isfile(CK):
    import gdown; gdown.download(id="1F6eVrR-0LseE-tfbT8Y8WT92_O11lt5o", output=CK, quiet=False)
sd = torch.load(CK, map_location="cpu"); sd = sd.get("model_state_dict", sd.get("state_dict", sd))
print("ckpt keys:", len(sd), list(sd)[:6], "...", [k for k in sd if "seq" in k or "aggregation" in k][:10], flush=True)
fcw = [v for k, v in sd.items() if k.endswith("aggregation.3.weight")][0]
JIST_FC = int(fcw.shape[0]); JIST_P = float([v for k, v in sd.items() if k.endswith("seq_gem.p")][0].reshape(-1)[0]) if any(k.endswith("seq_gem.p") for k in sd) else 3.0
print("JIST fc_output_dim", JIST_FC, "| seq_gem p", JIST_P, flush=True)
def _try_jist():
    m = torch.hub.load("gmberton/cosplace", "get_trained_model", backbone="ResNet18", fc_output_dim=JIST_FC, trust_repo=True)
    sub = {k[len("model."):]: v for k, v in sd.items() if k.startswith("model.")}
    mi, un = m.load_state_dict(sub, strict=False); print("JIST load missing", len(mi), mi[:5], "unexpected", len(un), un[:5], flush=True)
    assert len(mi) == 0, "JIST weights did not load"
    m.eval().to(DEVICE)
    @torch.no_grad()
    def fwd(x): return m(x)
    return fwd
REGISTRY["jist"] = _try_jist


## Dataset loaders (paths + poses). Every loader returns `{"parts": [(tag, paths)], "meta": {...}}`.

In [ ]:
# === L.RC — RobotCar-Seasons v2: overcast-reference rear map (NVM) + train queries =========
def _walk_find(root, name, max_depth=4):
    for d, subs, files in os.walk(root):
        if d.count(os.sep) - root.count(os.sep) >= max_depth: subs[:] = []
        if name in files: return os.path.join(d, name)
    return None

def load_robotcar():
    RC = PATHS["robotcar"]; assert RC, "robotcar mount not found"
    tr_txt = _walk_find(RC, "robotcar_v2_train.txt") or _walk_find("/kaggle/input", "robotcar_v2_train.txt", 6)
    nvm = os.path.join(RC, "3D-models", "all-merged", "all.nvm")
    if not os.path.isfile(nvm): nvm = _walk_find(RC, "all.nvm", 5)
    print("train.txt:", tr_txt, "| nvm:", nvm, flush=True)
    idx = {}
    for d, _, files in os.walk(os.path.join(RC, "images")):
        cam = os.path.basename(d)
        for f in files:
            if f.lower().endswith((".jpg", ".png")): idx[(cam, os.path.splitext(f)[0])] = os.path.join(d, f)
    print("image index:", len(idx), flush=True)
    # reference poses from the NVM camera block (quaternion = world->camera rotation, then camera centre)
    ref = []
    with open(nvm) as fh:
        fh.readline(); line = fh.readline()
        while line.strip() == "": line = fh.readline()
        n = int(line.split()[0])
        for _ in range(n):
            t = fh.readline().split(); parts = t[0].replace("\\", "/").split("/")
            cond, cam, ts = parts[-3], parts[-2], os.path.splitext(parts[-1])[0]
            if cond != "overcast-reference" or cam != "rear": continue
            q = np.array(list(map(float, t[2:6]))); C = np.array(list(map(float, t[6:9])))
            p = idx.get((cam, ts))
            if p: ref.append((t[0], p, C, quat_to_R(q).T, float(t[1])))
    print("NVM cameras:", n, "| overcast-reference/rear resolved:", len(ref), flush=True)
    qry = []
    for ln in open(tr_txt):
        t = ln.split()
        if len(t) < 17: continue
        M = np.array(list(map(float, t[1:17]))).reshape(4, 4)        # camera -> world (verified)
        cond, cam, ts = t[0].split("/")[0], t[0].split("/")[-2], os.path.splitext(t[0].split("/")[-1])[0]
        p = idx.get((cam, ts))
        if p: qry.append((t[0], p, M[:3, 3].copy(), M[:3, :3].copy(), cond))
    print("train queries resolved:", len(qry), "/ 1906", flush=True)
    meta = {"ref_names": np.array([r[0] for r in ref]), "ref_C": np.stack([r[2] for r in ref]),
            "ref_R": np.stack([r[3] for r in ref]), "ref_focal": np.array([r[4] for r in ref]),
            "q_names": np.array([q[0] for q in qry]), "q_C": np.stack([q[2] for q in qry]),
            "q_R": np.stack([q[3] for q in qry]), "q_cond": np.array([q[4] for q in qry])}
    # sanity: same world frame?  nearest reference centre per query + its orientation gap
    D = np.sqrt(((meta["q_C"][:, None, :] - meta["ref_C"][None, :, :]) ** 2).sum(-1)); nn = D.argmin(1)
    ang = np.array([rot_angle_deg(meta["q_R"][i], meta["ref_R"][nn[i]]) for i in range(len(nn))])
    log("L.RC.sanity", "ok" if np.median(D.min(1)) < 5 else "warn",
        f"query->nearest-ref dist median {np.median(D.min(1)):.2f} m (p90 {np.percentile(D.min(1),90):.2f}); "
        f"orientation gap median {np.median(ang):.1f} deg (p90 {np.percentile(ang,90):.1f})")
    return {"parts": [("ref", [r[1] for r in ref]), ("q", [q[1] for q in qry])], "meta": meta}

In [ ]:
# === L.CMU — Extended-CMU-Seasons: official database + dated query GT, camera c0 ===========
def _cmu_bases():
    out = {}
    for r in glob.glob("/kaggle/input/*"):
        for pat in ["slice*", "*/slice*", "*/*/slice*"]:
            for d in glob.glob(os.path.join(r, pat)):
                nm = os.path.basename(d)
                if not (os.path.isdir(d) and nm.startswith("slice") and nm[5:].isdigit()): continue
                inner = os.path.join(d, nm); b = inner if os.path.isdir(inner) else d
                if os.path.isdir(os.path.join(b, "camera-poses")): out.setdefault(int(nm[5:]), b)
    return out

def _read_poses(txt, cam="_c0_"):
    rows = []
    for ln in open(txt):
        t = ln.split()
        if len(t) < 8 or cam not in t[0]: continue
        try: v = [float(x) for x in t[1:8]]
        except ValueError: continue
        rows.append((t[0], np.array(v[4:7]), quat_to_R(v[:4]).T))     # centre, R camera->world
    return rows

def load_cmu():
    bases = _cmu_bases(); print("CMU slices with camera-poses:", sorted(bases), flush=True)
    parts, meta, total = [], {}, 0
    for s in CONFIG.CMU_SLICE_ORDER:
        if s not in bases: continue
        b = bases[s]; dbt = os.path.join(b, f"ground-truth-database-images-slice{s}.txt")
        if not os.path.isfile(dbt): continue
        idx = {}
        for sub in ["database", "query"]:
            for d, _, files in os.walk(os.path.join(b, sub)):
                for f in files: idx[f] = os.path.join(d, f)
        db = [(n, idx[n], C, R) for n, C, R in _read_poses(dbt) if n in idx]
        qs = []
        for pf in sorted(glob.glob(os.path.join(b, "camera-poses", "*.txt"))):
            date = "".join(ch for ch in os.path.basename(pf) if ch.isdigit())[-8:]
            qs += [(n, idx[n], C, R, date) for n, C, R in _read_poses(pf) if n in idx]
        if len(db) < 50 or len(qs) < 100: continue
        if total + len(db) + len(qs) > CONFIG.CMU_MAX_IMAGES: break
        total += len(db) + len(qs)
        parts += [(f"s{s}_db", [x[1] for x in db]), (f"s{s}_q", [x[1] for x in qs])]
        meta[f"s{s}"] = {"db_names": np.array([x[0] for x in db]), "db_C": np.stack([x[2] for x in db]),
                         "db_R": np.stack([x[3] for x in db]), "q_names": np.array([x[0] for x in qs]),
                         "q_C": np.stack([x[2] for x in qs]), "q_R": np.stack([x[3] for x in qs]),
                         "q_cond": np.array([x[4] for x in qs])}
        dmin = np.sqrt(((meta[f"s{s}"]["q_C"][:, None] - meta[f"s{s}"]["db_C"][None]) ** 2).sum(-1)).min(1)
        print(f"  slice{s}: db={len(db)} q={len(qs)} dates={len(set(x[4] for x in qs))} "
              f"| q->nearest-db median {np.median(dmin):.2f} m", flush=True)
    log("L.CMU", "ok" if meta else "fail", f"{len(meta)} slices, {total} images")
    return {"parts": parts, "meta": meta}

In [ ]:
# === L.NORD — Nordland 4 seasons, frame-aligned, uniform subsample ========================
def load_nordland():
    root = PATHS["nordland"]; assert root, "nordland mount not found"
    def season_dir(s):
        c = [d for d, _, fs in os.walk(root) if os.path.basename(d).lower() == s and any(f.lower().endswith((".png", ".jpg")) for f in fs)]
        return sorted(c, key=lambda p: -p.count(os.sep))[0] if c else None
    def fid(f):
        d = "".join(ch for ch in os.path.splitext(f)[0] if ch.isdigit()); return int(d) if d else None
    tracks = {}
    for s in ["summer", "spring", "fall", "winter"]:
        d = season_dir(s)
        if d: tracks[s] = {fid(f): os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith((".png", ".jpg")) and fid(f) is not None}
    common = sorted(set.intersection(*[set(v) for v in tracks.values()]))
    sel = np.unique(np.linspace(0, len(common) - 1, CONFIG.NORD_PER_SEASON).round().astype(int))
    fids = [common[i] for i in sel]
    log("L.NORD", "ok", f"seasons={list(tracks)} common={len(common)} selected={len(fids)} (stride~{len(common)/len(fids):.1f})")
    return {"parts": [(s, [tracks[s][f] for f in fids]) for s in tracks], "meta": {"frame_ids": np.array(fids), "seasons": np.array(list(tracks))}}

## Run: load → extract per descriptor → save (+ quick retrieval sanity)

In [ ]:
# === R.0 — load selected datasets ==========================================================
LOADERS = {"rc": load_robotcar, "cmu": load_cmu, "nord": load_nordland}
LOADED = {}
for key in DUMP:
    t0 = time.time()
    try: LOADED[key] = LOADERS[key](); log(f"R.0.{key}", "ok", f"{sum(len(p) for _, p in LOADED[key]['parts'])} images, {time.time()-t0:.0f}s")
    except Exception as e:
        import traceback; traceback.print_exc(); log(f"R.0.{key}", "fail", repr(e)[:200])
os.makedirs(os.path.join(CONFIG.OUT_DIR, "meta"), exist_ok=True)
def _save_meta(key, meta):
    if key == "cmu":
        for s, m in meta.items(): np.savez(os.path.join(CONFIG.OUT_DIR, "meta", f"cmu_{s}.npz"), **m)
    else: np.savez(os.path.join(CONFIG.OUT_DIR, "meta", f"{key}.npz"), **meta)
for key, D in LOADED.items(): _save_meta(key, D["meta"])
print("meta saved:", os.listdir(os.path.join(CONFIG.OUT_DIR, "meta")), flush=True)

In [ ]:
# === R.1 — extract + save (one pass over the images, all descriptors per batch: I/O is the bottleneck) ===
FEATS = {}
FWDS = {}
for dname in DESCRIPTORS:
    try: FWDS[dname] = build_extractor(dname)
    except Exception as e:
        import traceback; traceback.print_exc(); log(f"R.1.{dname}", "fail", repr(e)[:200])
DIMS = {}
for dname, fwd in FWDS.items():
    with torch.no_grad(): DIMS[dname] = int(fwd(torch.zeros(2, 3, CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, device=DEVICE)).shape[1])
print("descriptors ready:", DIMS, flush=True)
for dname in FWDS: os.makedirs(os.path.join(CONFIG.OUT_DIR, dname), exist_ok=True)

@torch.no_grad()
def extract_multi(paths, tag):
    dl = torch.utils.data.DataLoader(ImageList(paths, CONFIG.IMG_SIZE), batch_size=CONFIG.BATCH_SIZE, shuffle=False,
                                     num_workers=CONFIG.NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    outs = {d: np.empty((len(paths), DIMS[d]), np.float16) for d in FWDS}; t0 = time.time()
    amp = torch.bfloat16 if CONFIG.USE_BF16 else torch.float16
    for b, (x, idx, ok) in enumerate(dl):
        x = x.to(DEVICE, non_blocking=True)
        for d, fwd in FWDS.items():
            with torch.autocast("cuda", dtype=amp, enabled=(DEVICE == "cuda" and d != "netvlad")):
                y = fwd(x)
            outs[d][idx.numpy()] = torch.nn.functional.normalize(y.float(), dim=1).cpu().numpy().astype(np.float16)
        if b % 20 == 0: print(f"    {tag}: {min((b+1)*CONFIG.BATCH_SIZE, len(paths))}/{len(paths)} ({(b+1)*CONFIG.BATCH_SIZE/max(time.time()-t0,1e-3):.1f} img/s)", flush=True)
    dt = time.time() - t0; THROUGHPUT.append((tag, len(paths), dt))
    for d, a in outs.items():
        assert np.isfinite(a.astype(np.float32)).all(), f"non-finite {d} {tag}"
    log(f"E.1.{tag}", "ok", f"{len(paths)} imgs x {len(FWDS)} descriptors in {dt:.0f}s ({len(paths)/max(dt,1e-3):.1f} img/s)")
    return outs

for key, D in LOADED.items():
    for tag, paths in D["parts"]:
        outs = extract_multi(paths, f"{key}_{tag}")
        for dname, arr in outs.items():
            np.save(os.path.join(CONFIG.OUT_DIR, dname, f"{key}_{tag}.npy"), arr); FEATS[(dname, key, tag)] = arr

In [ ]:
# === R.2 — retrieval sanity with official thresholds (top-1 reference pose) ===============
THR = [(0.25, 2.0), (0.5, 5.0), (5.0, 10.0)]
def recall_at(qC, qR, rC, rR, sim):
    top = sim.argmax(1); dt = np.linalg.norm(qC - rC[top], axis=1)
    da = np.array([rot_angle_deg(qR[i], rR[top[i]]) for i in range(len(top))])
    return [float(np.mean((dt <= t) & (da <= a))) for t, a in THR], dt
SANITY = {}
for dname in FWDS:
    if "rc" in LOADED and (dname, "rc", "ref") in FEATS:
        m = LOADED["rc"]["meta"]; S = FEATS[(dname, "rc", "q")].astype(np.float32) @ FEATS[(dname, "rc", "ref")].astype(np.float32).T
        for c in sorted(set(m["q_cond"])):
            i = m["q_cond"] == c; r, dt = recall_at(m["q_C"][i], m["q_R"][i], m["ref_C"], m["ref_R"], S[i])
            SANITY[f"{dname}/rc/{c}"] = r; print(f"  {dname} RobotCar {c:16s} n={i.sum():4d} R@1 (.25m,2)/(.5m,5)/(5m,10) = {r[0]:.3f}/{r[1]:.3f}/{r[2]:.3f} | median err {np.median(dt):.1f} m", flush=True)
    if "cmu" in LOADED:
        for s, m in LOADED["cmu"]["meta"].items():
            if (dname, "cmu", f"{s}_db") not in FEATS: continue
            S = FEATS[(dname, "cmu", f"{s}_q")].astype(np.float32) @ FEATS[(dname, "cmu", f"{s}_db")].astype(np.float32).T
            r, dt = recall_at(m["q_C"], m["q_R"], m["db_C"], m["db_R"], S); SANITY[f"{dname}/cmu/{s}"] = r
            print(f"  {dname} CMU {s:5s} n={len(dt):5d} R@1 = {r[0]:.3f}/{r[1]:.3f}/{r[2]:.3f} | median err {np.median(dt):.1f} m", flush=True)
    if "nord" in LOADED and (dname, "nord", "summer") in FEATS:
        ref = FEATS[(dname, "nord", "summer")].astype(np.float32)
        for s in ["spring", "fall", "winter"]:
            if (dname, "nord", s) not in FEATS: continue
            top = (FEATS[(dname, "nord", s)].astype(np.float32) @ ref.T).argmax(1); err = np.abs(top - np.arange(len(top)))
            SANITY[f"{dname}/nord/{s}"] = [float(np.mean(err <= k)) for k in (0, 1, 2)]
            print(f"  {dname} Nordland {s:7s} R@1 (0/1/2 frames) = " + "/".join(f"{np.mean(err<=k):.3f}" for k in (0, 1, 2)), flush=True)

In [ ]:
# === R.3 — summary =========================================================================
summary = {"report": REPORT, "throughput": THROUGHPUT, "sanity": SANITY, "dump": DUMP, "descriptors": DESCRIPTORS,
           "files": sorted(os.path.relpath(os.path.join(d, f), CONFIG.OUT_DIR) for d, _, fs in os.walk(CONFIG.OUT_DIR) for f in fs)}
json.dump(summary, open(os.path.join(CONFIG.OUT_DIR, "summary.json"), "w"), indent=1, default=str)
tot = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(CONFIG.OUT_DIR) for f in fs)
print(f"DUMP_DONE files={len(summary['files'])} size={tot/1e6:.0f} MB", flush=True)
print("NB15_JSON=" + json.dumps({"sanity": SANITY, "fails": [k for k, v in REPORT.items() if v['status'] == 'fail']}), flush=True)